In [3]:
"""
train.py

Train an XGBoost model using feature_engineered_* files
produced by feature_engineering.py, mirroring 05_XGBoost.ipynb.
"""

from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor


PROC_DIR = Path("data/processed")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def train_model(
    train_path: Path = PROC_DIR / "feature_engineered_train.csv",
    eval_path: Path = PROC_DIR / "feature_engineered_eval.csv",
    target: str = "Life Expectancy",
    model_output: Path = ARTIFACT_DIR / "model_xgb.pkl",
):
    """
    Train XGBoost on feature_engineered_* data, as in 05_XGBoost.
    """

    train_df = pd.read_csv(train_path)
    eval_df = pd.read_csv(eval_path)

    X_train = train_df.drop(columns=[target])
    y_train = train_df[target]

    X_eval = eval_df.drop(columns=[target])
    y_eval = eval_df[target]

    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_eval)

    rmse = mean_squared_error(y_eval, preds)
    mae = mean_absolute_error(y_eval, preds)
    r2 = r2_score(y_eval, preds)

    print("=== Eval metrics (XGBoost) ===")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"R²:   {r2:.4f}")

    joblib.dump(model, model_output)
    print(f"✅ Saved model to {model_output}")

    return model, {"rmse": rmse, "mae": mae, "r2": r2}


if __name__ == "__main__":
    train_model()


=== Eval metrics (XGBoost) ===
RMSE: 2.6837
MAE:  1.1881
R²:   0.7570
✅ Saved model to artifacts/model_xgb.pkl
